In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NL = chr(10)

# 1. Загрузка исторических курсов Clio Infra (1870-2013, T=144)
xl_fx = pd.ExcelFile('ExchangeRatestoUKPound_Compact.xlsx')
df_long = xl_fx.parse('Data Long Format')
df_long['year'] = pd.to_numeric(df_long['year'], errors='coerce')
df_long['value'] = pd.to_numeric(df_long['value'], errors='coerce')

sub_all = df_long[(df_long['year'].between(1870, 2013)) & (df_long['value'] > 0)].dropna(subset=['value'])
piv_all = sub_all.pivot_table(index='year', columns='country.name', values='value').reindex(range(1870, 2014))

COUNTRIES_21 = [
    'United Kingdom', 'Switzerland', 'Sweden', 'Finland', 'Portugal', 'Italy',
    'France', 'Netherlands', 'Spain', 'Belgium', 'Denmark', 'Austria',
    'Australia', 'New Zealand', 'Canada', 'Japan', 'India',
    'Argentina', 'Brazil', 'Chile', 'Mexico'
]
COUNTRIES_20_GBP = [c for c in COUNTRIES_21 if c != 'United Kingdom']

piv_gbp = piv_all[COUNTRIES_20_GBP].interpolate(method='linear', limit_direction='both')
usd_ser = piv_all['United States'].interpolate(method='linear', limit_direction='both')
piv_usd = piv_all[COUNTRIES_21].interpolate(method='linear', limit_direction='both').div(usd_ser, axis=0)

def safe_zscore(M_arr):
    mean_v = np.mean(M_arr, axis=0)
    std_v = np.std(M_arr, axis=0, ddof=1)
    safe_std = np.where(std_v > 1e-12, std_v, 1.0)
    Z = (M_arr - mean_v) / safe_std
    Z[:, std_v <= 1e-12] = 0.0
    return Z

# 2. Калибровка 4 монетарных режимов и 95% бутстрэп-интервалы (Таблица 1)
dln_usd = np.diff(np.log(piv_usd.to_numpy(dtype=float)), axis=0)
dln_gbp = np.diff(np.log(piv_gbp.to_numpy(dtype=float)), axis=0)
yrs = np.arange(1871, 2014)

regimes_cfg = [
    ("Classical Gold Standard (1871-1913)", 1871, 1913),
    ("Interwar and Depression (1919-1939)", 1919, 1939),
    ("Bretton Woods Peg (1946-1971)", 1946, 1971),
    ("Post-Bretton Woods Float (1973-2013)", 1973, 2013)
]

print("=== TABLE 1: GLOBAL CURRENCY REGIMES CALIBRATION ===")
rng = np.random.default_rng(42)

for r_name, y0, y1 in regimes_cfg:
    m = (yrs >= y0) & (yrs <= y1)
    Xu = dln_usd[m, :]
    Zu = safe_zscore(Xu)
    Mu, Nu = Zu.shape
    Uu, su, Vtu = np.linalg.svd(Zu, full_matrices=False)
    u1u, v1u = np.take(Uu, 0, axis=1), np.take(Vtu, 0, axis=0)
    var1u = (su.item(0)**2) / np.sum(su**2) * 100.0
    pru = (1.0 / np.sum(u1u**4) - 1.0) / (Mu - 1.0)
    prv = (1.0 / np.sum(v1u**4) - 1.0) / (Nu - 1.0)

    sub_m = int(np.floor(Mu * 0.8))
    bu, bv = [], []
    for _ in range(300):
        idx = np.sort(rng.choice(Mu, size=sub_m, replace=False))
        Ub, _, Vtb = np.linalg.svd(safe_zscore(Xu[idx, :]), full_matrices=False)
        u1b, v1b = np.take(Ub, 0, axis=1), np.take(Vtb, 0, axis=0)
        bu.append((1.0 / np.sum(u1b**4) - 1.0) / (sub_m - 1.0))
        bv.append((1.0 / np.sum(v1b**4) - 1.0) / (Nu - 1.0))

    Xg = dln_gbp[m, :]
    Zg = safe_zscore(Xg)
    Ug, _, _ = np.linalg.svd(Zg, full_matrices=False)
    u1g = np.take(Ug, 0, axis=1)
    prug = (1.0 / np.sum(u1g**4) - 1.0) / (len(u1g) - 1.0)
    print(f"{r_name:<38} | M={Mu:2d} | Var1={var1u:4.1f}% | PR(u1)={pru:.3f} [{np.percentile(bu, 2.5):.3f}, {np.percentile(bu, 97.5):.3f}] | PR(v1)={prv:.3f} [{np.percentile(bv, 2.5):.3f}, {np.percentile(bv, 97.5):.3f}] | GBP_PR(u1)={prug:.3f}")

# 3. Отрисовка публикационных графиков (300 dpi)
fig1 = plt.figure(figsize=(15.5, 10.5), dpi=300)
plt.subplots_adjust(hspace=0.38, wspace=0.26)

ax1 = fig1.add_subplot(2, 2, 1)
m_iw = (yrs >= 1919) & (yrs <= 1939)
y_iw = yrs[m_iw]
U_iw, _, _ = np.linalg.svd(safe_zscore(dln_usd[m_iw, :]), full_matrices=False)
ax1.bar(y_iw, np.take(U_iw, 0, axis=1)**2, color='#1f4e78', width=0.7)
ax1.set_title("A. Interwar & Great Depression (1919-1939): Var1=46.8%, PR(u1)=0.247", fontsize=10.2, fontweight='bold')
ax1.set_ylabel("Squared Temporal Score u1^2", fontsize=9.5)
ax1.set_ylim(0, 0.34)
ax1.annotate('1920 Post-WWI' + NL + 'Deflation (26.4%)', xy=(1920, 0.264), xytext=(1923.5, 0.28), arrowprops=dict(arrowstyle='->', color='#800000', lw=0.8), fontsize=8, fontweight='bold', color='#800000')
ax1.annotate('1932-1934 Gold Bloc' + NL + 'Breakup (37.5%)', xy=(1933, 0.145), xytext=(1932, 0.23), arrowprops=dict(arrowstyle='->', color='#800000', lw=0.8), ha='center', fontsize=8, fontweight='bold', color='#800000')
ax1.grid(True, axis='y', linestyle='--', alpha=0.4)

ax2 = fig1.add_subplot(2, 2, 2)
m_bw = (yrs >= 1946) & (yrs <= 1971)
y_bw = yrs[m_bw]
U_bw, _, _ = np.linalg.svd(safe_zscore(dln_usd[m_bw, :]), full_matrices=False)
ax2.bar(y_bw, np.take(U_bw, 0, axis=1)**2, color='#385723', width=0.7)
ax2.set_title("B. Bretton Woods Peg (1946-1971): Var1=38.2%, PR(u1)=0.060", fontsize=10.2, fontweight='bold')
ax2.set_ylabel("Squared Temporal Score u1^2", fontsize=9.5)
ax2.set_ylim(0, 0.68)
ax2.annotate('1949-1950 Sterling &' + NL + 'European Devaluations (75.2%)', xy=(1950, 0.575), xytext=(1958, 0.54), arrowprops=dict(arrowstyle='->', color='#800000', lw=0.8), ha='center', fontsize=8, fontweight='bold', color='#800000')
ax2.grid(True, axis='y', linestyle='--', alpha=0.4)

ax3 = fig1.add_subplot(2, 2, 3)
m_fl = (yrs >= 1973) & (yrs <= 2013)
y_fl = yrs[m_fl]
U_fl, _, Vt_fl = np.linalg.svd(safe_zscore(dln_usd[m_fl, :]), full_matrices=False)
ax3.bar(y_fl, np.take(U_fl, 0, axis=1)**2, color='#c00000', width=0.7)
ax3.set_title("C. Post-Bretton Woods & Euro (1973-2013): Var1=35.6%, PR(u1)=0.043", fontsize=10.2, fontweight='bold')
ax3.set_ylabel("Squared Temporal Score u1^2", fontsize=9.5)
ax3.set_ylim(0, 0.70)
ax3.annotate('1999 Euro Launch' + NL + 'Denomination Step (59.7%)', xy=(1999, 0.597), xytext=(1988, 0.56), arrowprops=dict(arrowstyle='->', color='#800000', lw=0.8), ha='center', fontsize=8, fontweight='bold', color='#800000')
ax3.grid(True, axis='y', linestyle='--', alpha=0.4)

ax4 = fig1.add_subplot(2, 2, 4)
v1_fl_sq = (np.take(Vt_fl, 0, axis=0)**2) * 100.0
s_top10 = pd.Series(v1_fl_sq, index=COUNTRIES_21).sort_values(ascending=False).head(10)
EURO_SET = {'Netherlands', 'France', 'Finland', 'Belgium', 'Spain', 'Portugal', 'Italy', 'Austria'}
cols_10 = ['#c00000' if c in EURO_SET else '#595959' for c in s_top10.index]
x_10 = np.arange(len(s_top10))
ax4.bar(x_10, s_top10.values, color=cols_10)
ax4.axhline(100.0 / 21.0, color='black', linestyle=':', lw=1.1, label='Uniform Benchmark (1/21 = 4.76%)')
ax4.text(3.0, 13.0, 'Top 7 Eurozone Contributors = 73.7%' + NL + 'of All Mode 1 Energy (USD Base)', ha='center', fontsize=8.3, fontweight='bold', color='#800000')
ax4.set_xticks(x_10)
ax4.set_xticklabels(s_top10.index, rotation=38, ha='right', fontsize=8.2)
ax4.set_ylim(0, 15.2)
ax4.set_ylabel("Share of Mode 1 Spatial Energy (%)", fontsize=9.5)
ax4.set_title("D. Eurozone Bloc Identification in Post-1973 Era (Top 10 Countries)", fontsize=10.2, fontweight='bold')
ax4.legend(loc='upper right', fontsize=8)
ax4.grid(True, axis='y', linestyle='--', alpha=0.4)

fig1.suptitle("From Gold Standard to the Euro: Calibration of Spectral Diagnostics Across Global Monetary Regimes (1871-2013)", fontsize=12.0, fontweight='bold', y=0.98)
plt.savefig('Figure_1_ModernFX.png', bbox_inches='tight')
plt.close(fig1)

fig2 = plt.figure(figsize=(15.5, 6.8), dpi=300)
plt.subplots_adjust(wspace=0.26)

ax1 = fig2.add_subplot(1, 2, 1)
reg_short = ['Gold Standard' + NL + '(1871-1913)', 'Interwar' + NL + '(1919-1939)', 'Bretton Woods' + NL + '(1946-1971)', 'Post-BW Float' + NL + '(1973-2013)']
pru_usd_vals = [0.222, 0.247, 0.060, 0.043]
pru_gbp_vals = [0.172, 0.336, 0.008, 0.010]
prv_usd_vals = [0.387, 0.735, 0.696, 0.534]
prv_gbp_vals = [0.334, 0.438, 0.400, 0.390]

x_rg = np.arange(len(reg_short))
w_g = 0.20
ax1.bar(x_rg - 1.5*w_g, pru_usd_vals, width=w_g, color='#1f4e78', label='Temporal PR(u1) — USD Base')
ax1.bar(x_rg - 0.5*w_g, pru_gbp_vals, width=w_g, color='#8eaadb', label='Temporal PR(u1) — GBP Base')
ax1.bar(x_rg + 0.5*w_g, prv_usd_vals, width=w_g, color='#c00000', label='Spatial PR(v1) — USD Base')
ax1.bar(x_rg + 1.5*w_g, prv_gbp_vals, width=w_g, color='#f4b183', label='Spatial PR(v1) — GBP Base')
ax1.set_xticks(x_rg)
ax1.set_xticklabels(reg_short, fontsize=8.5)
ax1.set_ylim(0, 0.85)
ax1.set_ylabel("Normalized Participation Ratio (PR)", fontsize=9.5)
ax1.set_title("A. Numeraire Invariance: USD vs. Direct Sterling (GBP) Base", fontsize=10.2, fontweight='bold')
ax1.legend(loc='upper left', fontsize=7.8)
ax1.grid(True, axis='y', linestyle='--', alpha=0.4)

ax2 = fig2.add_subplot(1, 2, 2)
v1_fl = np.take(Vt_fl, 0, axis=0)
v2_fl = np.take(Vt_fl, 1, axis=0)
if v1_fl.item(6) < 0:
    v1_fl = -v1_fl
if v2_fl.item(17) < 0:
    v2_fl = -v2_fl

euro_core = {'Netherlands', 'France', 'Finland', 'Belgium', 'Spain', 'Portugal', 'Italy'}
other_eur = {'Austria', 'Sweden', 'Denmark', 'Switzerland', 'United Kingdom'}
lat_am = {'Argentina', 'Brazil', 'Chile', 'Mexico'}

for idx_c, cname in enumerate(COUNTRIES_21):
    xv, yv = float(v1_fl.item(idx_c)), float(v2_fl.item(idx_c))
    cpt = '#c00000' if cname in euro_core else ('#1f4e78' if cname in other_eur else ('#385723' if cname in lat_am else '#595959'))
    sz = 85 if cname in euro_core else 65
    ax2.scatter([xv], [yv], color=cpt, s=sz, zorder=5)

lbl_21 = [
    ('Netherlands', 0.012, 0.025),
    ('France', 0.012, -0.035),
    ('Italy', 0.012, -0.02),
    ('Austria', -0.06, -0.045),
    ('United Kingdom', 0.012, 0.02),
    ('Japan', 0.012, -0.03),
    ('Argentina', 0.012, 0.01),
    ('Brazil', 0.012, -0.035)
]
for cname, dx, dy in lbl_21:
    idx_c = COUNTRIES_21.index(cname)
    xv, yv = float(v1_fl.item(idx_c)), float(v2_fl.item(idx_c))
    ax2.annotate(cname, xy=(xv, yv), xytext=(xv + dx, yv + dy), fontsize=8.2, fontweight='bold' if cname in ['Netherlands', 'France', 'Argentina', 'United Kingdom'] else 'normal', color='#222222')

ax2.axhline(0, color='black', linestyle=':', lw=0.9)
ax2.axvline(1.0 / np.sqrt(21.0), color='black', linestyle=':', lw=0.9, label='Uniform Mode 1 Loading (1/√21 = 0.218)')
ax2.text(0.28, 0.22, 'Core Eurozone Bloc' + NL + '(1999 Conversion Shock)', fontsize=8.3, fontweight='bold', color='#800000')
ax2.text(0.01, -0.18, 'Latin American' + NL + 'Debt-Crisis Pole', fontsize=8.3, fontweight='bold', color='#385723')
ax2.set_xlim(-0.04, 0.46)
ax2.set_ylim(-0.32, 0.56)
ax2.set_xlabel("Mode 1 Eurozone Conversion Loading (v1, Var1 = 35.6%)", fontsize=9.5)
ax2.set_ylabel("Mode 2 Emerging-Market Crisis Pole (v2, Var2 = 17.6%)", fontsize=9.5)
ax2.set_title("B. Two-Mode Structural Map of Global Currencies (1973-2013)", fontsize=10.2, fontweight='bold')
ax2.legend(loc='lower right', fontsize=8)
ax2.grid(True, linestyle='--', alpha=0.4)
fig2.suptitle("Calibration Robustness: Numeraire Invariance and Bloc Separation (1871-2013)", fontsize=12.4, fontweight='bold', y=1.01)
plt.savefig('Figure_2_ModernFX.png', bbox_inches='tight')
plt.close(fig2)
print("Clean modern FX pipeline completed!")

/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


=== TABLE 1: GLOBAL CURRENCY REGIMES CALIBRATION ===
Classical Gold Standard (1871-1913)    | M=43 | Var1=27.0% | PR(u1)=0.222 [0.182, 0.476] | PR(v1)=0.387 [0.372, 0.466] | GBP_PR(u1)=0.172
Interwar and Depression (1919-1939)    | M=21 | Var1=46.8% | PR(u1)=0.247 [0.151, 0.317] | PR(v1)=0.735 [0.645, 0.782] | GBP_PR(u1)=0.336
Bretton Woods Peg (1946-1971)          | M=26 | Var1=38.2% | PR(u1)=0.060 [0.010, 0.078] | PR(v1)=0.696 [0.362, 0.706] | GBP_PR(u1)=0.008
Post-Bretton Woods Float (1973-2013)   | M=41 | Var1=35.6% | PR(u1)=0.043 [0.028, 0.439] | PR(v1)=0.534 [0.434, 0.693] | GBP_PR(u1)=0.010
Clean modern FX pipeline completed!
